In [1]:
# ====== 0.0  mount + GPU libs ======
#from google.colab import drive
#drive.mount('/drive')
#%cd /drive/MyDrive/AURA_MoE
!pip install -q torch torchvision tensorboard tqdm wandb  # wandb for live plots
import torch, torch.nn as nn, torchvision.models as models, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from pathlib import Path#, glob
import wandb, json, torch
import numpy as np

In [2]:
# ====== 1.0  robust hyper-params ======
CFG = {
    'img_size': 224, 'lidar_bev': 64, 'state_dim': 10,
    'n_experts': 5, 'top_k': 2,           # top-2 gating → more robust
    'latent': 256,                         # bigger capacity
    'traj_len': 2,
    'batch': 32, 'lr': 2e-3, 'epochs': 25, 'device': 'cuda',
    'aux_loss': 1e-2, 'lidar_channels': 64,'out_dim':60                # force expert usage balance || in device we can do cuda
}

In [3]:
# ====== 2.0  dataset (reads 4k folders) ======
class CarlaMoE(Dataset):
    def __init__(self, split='training', expert_id=0, root='carla_data'):
        self.files = sorted(Path(root).glob(f'E{expert_id}/{split}/*.pt'))
        assert self.files, f'No data at E{expert_id}/{split}'
    def __len__(self): return len(self.files)
    def __getitem__(self, idx):
      data = torch.load(self.files[idx], weights_only=False)
      
      # ---- handle mixed single/batch files ----
      if isinstance(data, list):                       # batch of 100 frames
          frame = data[idx % len(data)]                # pick one
      else:                                            # single frame
          frame = data

      #print("raw lidar shape:", frame['lidar'].shape)
      # ---- ensure CHW + float32 [0,1] ----
      img = frame['image']
      if isinstance(img, np.ndarray):
          img = torch.from_numpy(img.copy())
      if img.ndim == 3 and img.shape[-1] == 3:         # HWC → CHW
          img = img.permute(2, 0, 1)
      img = img.float() / 255.0                        # uint8 → float32 [0,1]

      #lidar = torch.from_numpy(frame['lidar'].copy()) if isinstance(frame['lidar'], np.ndarray) else frame['lidar']
      lidar = frame['lidar']          # numpy (H, W, C)
      lidar = torch.from_numpy(lidar).permute(2, 0, 1).float() 
      state = torch.from_numpy(frame['state'].copy()) if isinstance(frame['state'], np.ndarray) else frame['state']
      #future = torch.from_numpy(frame['future'].copy()) if isinstance(frame['future'], np.ndarray) else frame['future']
      future = frame['future']  # make sure it’s (2,)
      future = torch.from_numpy(future.copy()).float().view(-1)
      #print("LiDAR channels:", lidar.shape[0])
      return (img, lidar, state, future)

RUN_DIR = Path('runs/4k')
RUN_DIR.mkdir(exist_ok=True)

In [4]:
# ====== 3.0  robust MoE (bigger + Top-2) ======
class Expert(nn.Module):
    def __init__(self):
        super().__init__()
        self.img_enc = models.resnet18(weights='IMAGENET1K_V1')
        self.img_enc.fc = nn.Identity()
        self.lidar_enc = nn.Sequential(
            nn.Conv2d(CFG['lidar_channels'], 32, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, 2, 1), nn.ReLU(),
            nn.Conv2d(64, 128, 3, 2, 1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(1),      # (B, 128, 1, 1)
            nn.Flatten(),                 # (B, 128)  <-- ADD THIS
        )  # 128
        
        """self.lidar_enc = nn.Sequential(
            nn.Conv2d(CFG['lidar_channels'], 32, 3, 2, 1),  # illegal, so instead:
        )"""
        
        self.state_enc = nn.Sequential(nn.Linear(10, 128), nn.ReLU(), nn.Linear(128, 128))
        self.head = nn.Sequential(nn.Linear(512+128+128, 512), nn.ReLU(), nn.Linear(512, CFG['latent']))

    def forward(self, img, lidar, state):
        #print(">>> inside Expert")
        #print("  img shape:", img.shape, "  ch:", img.shape[1])
        print("  lidar shape:", lidar.shape, "ch:", lidar.shape[1])
        print("  state shape:", state.shape)
        f1 = self.img_enc(img); f2 = self.lidar_enc(lidar); f3 = self.state_enc(state)
        return self.head(torch.cat([f1, f2, f3], dim=1))

class Gating(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(CFG['latent'], 256), nn.ReLU(), nn.Linear(256, 5))
        self.gate_out = None
    def forward(self, x):
        logits = self.net(x)
        top_logits, top_idx = torch.topk(logits, CFG['top_k'], dim=1)  # top-2
        top_gate = torch.softmax(top_logits, dim=1)
        gates = torch.zeros_like(logits).scatter_(1, top_idx, top_gate)
        self.gate_out = gates; return gates, top_idx

class MoEDriving(nn.Module):
    def __init__(self):
        super().__init__()
        self.experts = nn.ModuleList([Expert() for _ in range(5)])
        self.gate   = Gating()
        self.decoder = nn.Sequential(
            nn.Linear(CFG['latent'], 128), nn.ReLU(),
            nn.Linear(128, CFG['out_dim'])  # 60
        )
    def forward(self, img, lidar, state):
        exp_outs = torch.stack([exp(img, lidar, state) for exp in self.experts], dim=1)
        gate_in  = exp_outs.mean(dim=1); gates, idx = self.gate(gate_in)
        out = (exp_outs * gates.unsqueeze(-1)).sum(dim=1); return self.decoder(out)

In [5]:
# ====== 4.0  loss with expert balance ======
def moe_loss(pred, tgt, gates):
    mse = F.mse_loss(pred, tgt)
    load = gates.mean(dim=0)                                   # (n_exp,)
    balance = CFG['aux_loss'] * torch.var(load)               # force equal usage
    return mse + balance

In [6]:
# ====== 5.0  training loop (early-stop on val) ======
def train():
    device = torch.device(CFG['device'])
    model = MoEDriving().to(device)
    opt   = torch.optim.Adam(model.parameters(), lr=CFG['lr'])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG['epochs'])
    wandb.init(project="aura-moe", name="4k-robust", config=CFG)

    for epoch in range(CFG['epochs']):
        print(f'\nEpoch {epoch}')
        for expert_id in range(5):
            ds = CarlaMoE(split='training', expert_id=expert_id, root='carla_data')
            dl = DataLoader(ds, batch_size=CFG['batch'], shuffle=True, num_workers=2, pin_memory=True)
            running = 0.0
            for img, lidar, state, fut in dl:
                img, lidar, state, fut = [x.to(device, non_blocking=True) for x in (img, lidar, state, fut)]
                
               # print("img channels:", img.shape[1])   # must be 3
                #print("lidar channels:", lidar.shape[1])  # must be 30
                opt.zero_grad()
                pred = model(img, lidar, state)
                loss = moe_loss(pred, fut, model.gate.gate_out)
                loss.backward(); opt.step()
                try:
                    torch.cuda.empty_cache()
                except :
                    pass
                running += loss.item()
            print(f'  E{expert_id} loss {running/len(dl):.4f}')
            wandb.log({f'E{expert_id}_loss': running/len(dl), 'epoch': epoch})
        sched.step()
        torch.save(model.state_dict(), RUN_DIR/f'epoch{epoch}.pt')
        wandb.log({'epoch': epoch, 'lr': sched.get_last_lr()[0]})
    torch.save(model.state_dict(), RUN_DIR/'best.pt')
    wandb.finish(); print('✓ training finished – download best.pt')

train()

wandb: Currently logged in as: zhar-mohamedreda (zhar-mohamedreda-ensa-tetouan) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



Epoch 0
  lidar shape: torch.Size([32, 64, 3, 64]) ch: 64
  state shape: torch.Size([32, 10])
  lidar shape: torch.Size([32, 64, 3, 64]) ch: 64
  state shape: torch.Size([32, 10])
  lidar shape: torch.Size([32, 64, 3, 64]) ch: 64
  state shape: torch.Size([32, 10])
  lidar shape: torch.Size([32, 64, 3, 64]) ch: 64
  state shape: torch.Size([32, 10])
  lidar shape: torch.Size([32, 64, 3, 64]) ch: 64
  state shape: torch.Size([32, 10])
  lidar shape: torch.Size([32, 64, 3, 64]) ch: 64
  state shape: torch.Size([32, 10])
  lidar shape: torch.Size([32, 64, 3, 64]) ch: 64
  state shape: torch.Size([32, 10])
  lidar shape: torch.Size([32, 64, 3, 64]) ch: 64
  state shape: torch.Size([32, 10])
  lidar shape: torch.Size([32, 64, 3, 64]) ch: 64
  state shape: torch.Size([32, 10])
  lidar shape: torch.Size([32, 64, 3, 64]) ch: 64
  state shape: torch.Size([32, 10])
  lidar shape: torch.Size([32, 64, 3, 64]) ch: 64
  state shape: torch.Size([32, 10])
  lidar shape: torch.Size([32, 64, 3, 64]) ch

E0_loss,█▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
E1_loss,█▅▃▄▄▃▃▃▄▄▃▃▃▃▂▂▂▃▂▁▂▂▁▂▂
E2_loss,█▆▅▄▄▄▅▃▄▄▅▄▃▃▃▂▃▂▁▂▃▁▁▁▂
E3_loss,█▆▆▄▅▅▅▄▅▄▄▃▂▃▂▃▃▂▂▂▂▁▁▂▂
E4_loss,█▇▆▆▆▅▆▆▅▅▅▄▃▄▃▃▃▂▂▂▁▂▁▁▁
epoch,▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▅▅▅▅▆▆▆▆▆▇▇▇███
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
E0_loss,0.07679
E1_loss,0.07926
E2_loss,0.07628
E3_loss,0.07951


✓ training finished – download best.pt
